# Study 2: Pollution dynamics and mitigation

- This notebook explores a shared atmospheric CO2 stock as a calculation separate from the regional environment. The stock adds each region's emissions that remain after mitigation.
- I compare the current mitigation capacity and costs with one more demanding candidate. The candidate is for sensitivity analysis, not a real-world estimate.
- The environment now uses nonlinear gross emissions and mitigation only avoids new emissions. Saved outputs below are from the previous rules; rerun all cells before interpreting tables.

## Define the stock calculation

- The IPCC reports that about 46% of human CO2 emissions remained in the atmosphere during 2010-2019; land and ocean sinks took up the rest. I use 0.46 as a simple airborne fraction for new emissions, not as a fixed law. [IPCC AR6 WGI, Chapter 5](https://www.ipcc.ch/report/ar6/wg1/chapter/chapter-5/)
- `stock_scale` converts this project's arbitrary emissions into arbitrary atmospheric-stock units. No physical calibration is available yet.
- The prototype is `stock_next = stock_now + 0.46 * stock_scale * total_remaining_emissions`. It does not remove the stock already in the atmosphere. Mitigation reduces new emissions; it is not carbon removal.

In [42]:
import pandas as pd
import random


from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.agents import TabularQLearner
from study_2_economic_sustainability.config import TrainingConfig
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment

In [43]:
config = EnvironmentConfig()
AIRBORNE_FRACTION = 0.46
INITIAL_STOCK = 10.0
DOOM_THRESHOLD = 100.0
STOCK_SCALE = 0.10
EDI_LEVELS = (0, 70, 100)

### Compare mitigation settings

In [44]:
MITIGATION_SETTINGS = {
    "current": {
        "capacities": config.mitigation_capacity,
        "costs": config.action_costs,
    },
    "candidate": {
        "capacities": (0, 0, 4, 8, 12),
        "costs": (0, 100, 20, 50, 100),
    },
}
ACTION_NAMES = ("save", "develop", "low", "medium", "high")

- The candidate lowers high capacity from 30 to 12 and raises its cost from 50 to 100. Low and medium are adjusted too, so the three mitigation choices still increase in strength and cost.
- These values are deliberately labeled as a candidate. We are testing if the current action scale makes mitigation too easy

In [45]:
mitigation_rows = []
for setting_name, setting in MITIGATION_SETTINGS.items():
    for action_id, action_name in enumerate(ACTION_NAMES):
        mitigation_rows.append({
            "setting": setting_name,
            "action": action_name,
            "cost": setting["costs"][action_id],
            "capacity": setting["capacities"][action_id],
        })
display(pd.DataFrame(mitigation_rows).set_index(["setting", "action"]))

cost  capacity
setting   action                 
current   save        0         0
          develop   100         0
          low        10         5
          medium     25        15
          high       50        30
candidate save        0         0
          develop   100         0
          low        20         4
          medium     50         8
          high      100        12

## How fast does the stock reach 100?

- This calculation assumes all four regions remain at the same EDI and choose the same mitigation action every step. It isolates the pollution mechanics from learning and currency constraints.

In [46]:
def emissions_at_edi(edi):
    return (
        config.emissions_base
        + config.emissions_edi_linear * edi
        + config.emissions_edi_quadratic * edi ** 2
    )

def stock_steps_to_threshold(edi, capacity, stock_scale):
    emissions = emissions_at_edi(edi)
    remaining = max(0, emissions - capacity)
    stock_increase_per_step = (
        4 * AIRBORNE_FRACTION * stock_scale * remaining
    )
    if stock_increase_per_step == 0:
        return None
    stock_gap = DOOM_THRESHOLD - INITIAL_STOCK
    return stock_gap / stock_increase_per_step

In [47]:
stock_rows = []
for setting_name, setting in MITIGATION_SETTINGS.items():
    for edi in EDI_LEVELS:
        for action_id, action_name in enumerate(ACTION_NAMES):
            steps = stock_steps_to_threshold(
                edi, setting["capacities"][action_id], STOCK_SCALE
            )
            stock_rows.append({
                "setting": setting_name,
                "edi": edi,
                "action": action_name,
                "remaining_emissions_per_region": max(
                    0, emissions_at_edi(edi) - setting["capacities"][action_id]
                ),
                "steps_to_threshold": steps,
            })
stock_scenarios = pd.DataFrame(stock_rows)
display(stock_scenarios.set_index(["setting", "edi", "action"]))

remaining_emissions_per_region  steps_to_threshold
setting   edi action                                                     
current   0   save                                8.0           61.141304
              develop                             8.0           61.141304
              low                                 3.0          163.043478
              medium                              0.0                 NaN
              high                                0.0                 NaN
          70  save                               19.9           24.579419
              develop                            19.9           24.579419
              low                                14.9           32.827546
              medium                              4.9           99.822538
              high                                0.0                 NaN
          100 save                               28.0           17.468944
              develop                            28.0           17.468944
              low                                23.0           21.266541
              medium                             13.0           37.625418
              high                                0.0                 NaN
candidate 0   save                                8.0           61.141304
              develop                             8.0           61.141304
              low                                 4.0          122.282609
              medium                              0.0                 NaN
              high                                0.0                 NaN
          70  save                               19.9           24.579419
              develop                            19.9           24.579419
              low                                15.9           30.762920
              medium                             11.9           41.103398
              high                                7.9           61.915245
          100 save                               28.0           17.468944
              develop                            28.0           17.468944
              low                                24.0           20.380435
              medium                             20.0           24.456522
              high                               16.0           30.570652

- `None` means the chosen action offsets all new emissions in this fixed-EDI scenario, so the stock does not reach the threshold through new emissions.
- Under the current settings, medium mitigation offsets emissions at EDI 0 and 70, while high mitigation offsets them at all three EDI levels. At EDI 100, medium mitigation leaves a small residual, and the table estimates about 489 steps to reach 100.
- Under the candidate, high mitigation still offsets emissions at EDI 0, but leaves 1.6 units per region at EDI 70 and 4 at EDI 100. The estimated times are about 306 and 122 steps. This is a meaningful change in the toy calculation, though both times depend on the illustrative stock scale and fixed EDI assumptions.

## Check how much high mitigation costs to sustain

In [48]:
income_rows = []
for setting_name, setting in MITIGATION_SETTINGS.items():
    for edi in EDI_LEVELS:
        income = config.income_base + config.income_per_edi * edi
        high_cost = setting["costs"][4]
        income_rows.append({
            "setting": setting_name,
            "edi": edi,
            "income_per_step": income,
            "high_cost": high_cost,
            "balance_change_after_high": income - high_cost,
        })
display(pd.DataFrame(income_rows).set_index(["setting", "edi"]))

income_per_step  high_cost  balance_change_after_high
setting   edi                                                       
current   0               20.0         50                      -30.0
          70              55.0         50                        5.0
          100             70.0         50                       20.0
candidate 0               20.0        100                      -80.0
          70              55.0        100                      -45.0
          100             70.0        100                      -30.0

- With current costs, income covers the action at EDI 70 and 100, but not at EDI 0. With the candidate cost of 100, income does not cover it at any of these EDI levels
- That makes the candidate change two things at once: its lower capacity leaves more emissions, and its higher price limits how often agents can buy it.

## How do regions affect the shared stock?

- An extra unit of remaining emissions from any region adds the same amount to the shared stock. This is the climate externality: every region contributes to the same stock.
- There is no separate neighbor spillover coefficient for atmospheric CO2. The earlier neighbor rule describes a local pollutant moving between adjacent regions

In [49]:
EXTRA_EMISSIONS_FROM_ONE_REGION = 5
shared_stock_increase = (
    AIRBORNE_FRACTION * STOCK_SCALE * EXTRA_EMISSIONS_FROM_ONE_REGION
)
spillover_rows = []
for source_region in range(4):
    spillover_rows.append({
        "source_region": source_region,
        "global_stock_increase": shared_stock_increase,
        "increase_seen_by_each_region": shared_stock_increase,
    })
display(pd.DataFrame(spillover_rows).set_index("source_region"))

,global_stock_increase,increase_seen_by_each_region
source_region,,
0,0.23,0.23
1,0.23,0.23
2,0.23,0.23
3,0.23,0.23


- With the illustrative stock scale of 0.10, five extra emission units from any region add 0.23 stock units. The same shared increase affects every region; it is not divided among them. In this model, a region's emissions therefore affect all regions through the shared stock, rather than traveling only to its neighbors.

## Test mitigation capacity on its own

In [50]:
HIGH_CAPACITY_OPTIONS = (30, 20, 12, 8, 4)
CAPACITY_TEST_COST = config.action_costs[4]

capacity_rows = []
for capacity in HIGH_CAPACITY_OPTIONS:
    for edi in (70, 100):
        steps = stock_steps_to_threshold(edi, capacity, STOCK_SCALE)
        capacity_rows.append({
            "high_capacity": capacity,
            "high_cost_held_at": CAPACITY_TEST_COST,
            "edi": edi,
            "remaining_emissions_per_region": max(
                0, emissions_at_edi(edi) - capacity
            ),
            "steps_to_threshold": steps,
            "within_200_step_episode": (
                steps is not None and steps <= config.episode_steps
            ),
        })
display(pd.DataFrame(capacity_rows).set_index(["high_capacity", "edi"]))

high_cost_held_at  remaining_emissions_per_region  \
high_capacity edi                                                      
30            70                  50                             0.0   
              100                 50                             0.0   
20            70                  50                             0.0   
              100                 50                             8.0   
12            70                  50                             7.9   
              100                 50                            16.0   
8             70                  50                            11.9   
              100                 50                            20.0   
4             70                  50                            15.9   
              100                 50                            24.0   

                   steps_to_threshold  within_200_step_episode  
high_capacity edi                                               
30            70                  NaN                    False  
              100                 NaN                    False  
20            70                  NaN                    False  
              100           61.141304                     True  
12            70            61.915245                     True  
              100           30.570652                     True  
8             70            41.103398                     True  
              100           24.456522                     True  
4             70            30.762920                     True  
              100           20.380435                     True

- Higher capacity lowers the residual emissions from each region.
- With high capacity 12, the estimate is about 306 steps at EDI 70 and 122 at EDI 100. Under the fixed 200-step episode, only the EDI 100 case reaches the threshold. Capacity 8 or 4 reaches it within 200 steps at both EDI levels.
- Capacity 20 and 30 offset all new emissions at these EDI levels in this calculation, so the stock does not reach 100 through new emissions. This highlights why the current capacity of 30 can prevent accumulation in the prototype.

## Check sensitivity to the stock scale


In [51]:
STOCK_SCALE_OPTIONS = (0.05, 0.10, 0.20)
SCALE_TEST_CAPACITY = 12

scale_rows = []
for stock_scale in STOCK_SCALE_OPTIONS:
    for edi in (70, 100):
        scale_rows.append({
            "stock_scale": stock_scale,
            "high_capacity": SCALE_TEST_CAPACITY,
            "edi": edi,
            "steps_to_threshold": stock_steps_to_threshold(
                edi, SCALE_TEST_CAPACITY, stock_scale
            ),
        })
display(pd.DataFrame(scale_rows).set_index(["stock_scale", "edi"]))

high_capacity  steps_to_threshold
stock_scale edi                                   
0.05        70              12          123.830490
            100             12           61.141304
0.10        70              12           61.915245
            100             12           30.570652
0.20        70              12           30.957622
            100             12           15.285326

- At EDI 100 with capacity 12, the estimate ranges from about 245 steps at scale 0.05 to 61 steps at scale 0.20; doubling the scale halves the estimated time. At EDI 70 the same range is about 611 to 153 steps.
- This large change shows that the stock scale strongly controls whether the threshold is reachable within an episode.

## Explore a range of mitigation costs

- Hold mitigation capacities at the candidate levels and vary only the prices of low, medium, and high mitigation. This keeps the capacity question separate from the cost question.

In [52]:
MITIGATION_COST_MULTIPLIERS = (0.5, 1.0, 1.5, 2.0)
COST_TEST_CAPACITIES = (4, 8, 12)

cost_rows = []
for cost_multiplier in MITIGATION_COST_MULTIPLIERS:
    for action_id, action_name in enumerate(("low", "medium", "high"), start=2):
        action_cost = config.action_costs[action_id] * cost_multiplier
        for edi in (0, 70, 100):
            income = config.income_base + config.income_per_edi * edi
            cost_rows.append({
                "cost_multiplier": cost_multiplier,
                "action": action_name,
                "capacity_held_fixed": COST_TEST_CAPACITIES[action_id - 2],
                "edi": edi,
                "action_cost": action_cost,
                "income_per_step": income,
                "balance_change_after_action": income - action_cost,
            })
display(pd.DataFrame(cost_rows).set_index(["cost_multiplier", "action", "edi"]))

capacity_held_fixed  action_cost  income_per_step  \
cost_multiplier action edi                                                      
0.5             low    0                      4          5.0             20.0   
                       70                     4          5.0             55.0   
                       100                    4          5.0             70.0   
                medium 0                      8         12.5             20.0   
                       70                     8         12.5             55.0   
                       100                    8         12.5             70.0   
                high   0                     12         25.0             20.0   
                       70                    12         25.0             55.0   
                       100                   12         25.0             70.0   
1.0             low    0                      4         10.0             20.0   
                       70                     4         10.0             55.0   
                       100                    4         10.0             70.0   
                medium 0                      8         25.0             20.0   
                       70                     8         25.0             55.0   
                       100                    8         25.0             70.0   
                high   0                     12         50.0             20.0   
                       70                    12         50.0             55.0   
                       100                   12         50.0             70.0   
1.5             low    0                      4         15.0             20.0   
                       70                     4         15.0             55.0   
                       100                    4         15.0             70.0   
                medium 0                      8         37.5             20.0   
                       70                     8         37.5             55.0   
                       100                    8         37.5             70.0   
                high   0                     12         75.0             20.0   
                       70                    12         75.0             55.0   
                       100                   12         75.0             70.0   
2.0             low    0                      4         20.0             20.0   
                       70                     4         20.0             55.0   
                       100                    4         20.0             70.0   
                medium 0                      8         50.0             20.0   
                       70                     8         50.0             55.0   
                       100                    8         50.0             70.0   
                high   0                     12        100.0             20.0   
                       70                    12        100.0             55.0   
                       100                   12        100.0             70.0   

                            balance_change_after_action  
cost_multiplier action edi                               
0.5             low    0                           15.0  
                       70                          50.0  
                       100                         65.0  
                medium 0                            7.5  
                       70                          42.5  
                       100                         57.5  
                high   0                           -5.0  
                       70                          30.0  
                       100                         45.0  
1.0             low    0                           10.0  
                       70                          45.0  
                       100                         60.0  
                medium 0                           -5.0  
                       70                          30.0  
                     

- At the current price level (1.0x), high mitigation costs 50. Income is 55 at EDI 70 and 70 at EDI 100, so income alone can replenish that purchase at those development levels. At 1.5x, high mitigation costs 75 and exceeds income at both levels; low and medium remain below income at EDI 70 and 100.
- The 1.5x setting looks like a useful middle case to test: high mitigation would require saved currency at EDI 70 and 100, while lower mitigation actions remain more affordable. At 2.0x, the high action costs 100 and income does not cover it at any tested EDI. This is a hypothesis about affordability, not evidence that 1.5x is the best policy cost.

## Pilot learned policies under candidate settings

-  It holds the combined EDI and income reward, mixed reward mode, pollution dynamics, spillover, and failure threshold fixed.
- Four settings are compared: current values, reduced mitigation capacity, 1.5 times current mitigation costs, and both changes together. Each uses five training seeds and the same 20 held-out starts.

In [53]:
PILOT_TRAINING_SEEDS = (11, 23, 37, 41, 53)
PILOT_EVALUATION_SEEDS = tuple(range(100_000, 100_020))
PILOT_TRAINING_EPISODES = 2_000
PILOT_REWARD_MODE = "mixed"
pilot_training_config = TrainingConfig()

In [ ]:
higher_action_costs = []
for cost in config.action_costs:
    higher_action_costs.append(cost * 1.5)
higher_action_costs = tuple(higher_action_costs)

PILOT_SETTINGS = {
    "current": {
        "capacities": config.mitigation_capacity,
        "costs": config.action_costs,
    },
    "reduced_capacity": {
        "capacities": (0, 0, 4, 8, 12),
        "costs": config.action_costs,
    },
    "higher_cost": {
        "capacities": config.mitigation_capacity,
        "costs": higher_action_costs,
    },
    "reduced_capacity_and_higher_cost": {
        "capacities": (0, 0, 4, 8, 12),
        "costs": higher_action_costs,
    },
}

def make_pilot_config(setting):
    pilot_config = EnvironmentConfig()
    pilot_config.mitigation_capacity = setting["capacities"]
    pilot_config.action_costs = setting["costs"]
    pilot_config.spillover = setting.get("spillover", config.spillover)
    return pilot_config


In [ ]:
def pilot_actions(agents, observations, greedy):
    actions = []
    for region in range(4):
        action = agents[region].select_action(
            observations[region], greedy=greedy
        )
        actions.append(action)
    return tuple(actions)

def pilot_rewards(environment, info, terminated):
    local_rewards = []
    for region in range(4):
        income_bonus = (info["income"][region] - environment.config.income_base) / 10
        economic_reward = info["development_gains"][region] + income_bonus
        reward = environment.config.economy_weight * economic_reward
        reward += environment.config.pollution_weight * info["pollution_rewards"][region]
        if not info["valid_actions"][region]:
            reward -= environment.config.invalid_action_penalty
        if terminated:
            reward -= environment.config.failure_penalty
        local_rewards.append(reward)

    shared_reward = sum(local_rewards) / 4
    mixed_rewards = []
    for reward in local_rewards:
        mixed_rewards.append(0.7 * reward + 0.3 * shared_reward)
    return tuple(mixed_rewards)


In [ ]:
def train_pilot_policy(setting, training_seed):
    pilot_config = make_pilot_config(setting)
    environment = EconomicSustainabilityEnvironment(PILOT_REWARD_MODE, pilot_config)
    episode_rng = random.Random(training_seed)
    agents = []
    for region in range(4):
        agent = TabularQLearner(training_seed * 10 + region, pilot_training_config)
        agents.append(agent)

    for _ in range(PILOT_TRAINING_EPISODES):
        observations = environment.reset(episode_rng.randrange(100_000))
        while True:
            actions = pilot_actions(agents, observations, greedy=False)
            next_observations, _, terminated, truncated, info = environment.step(actions)
            rewards = pilot_rewards(environment, info, terminated)
            for region in range(4):
                agents[region].update(
                    observations[region], actions[region], rewards[region],
                    next_observations[region], terminated or truncated,
                )
            observations = next_observations
            if terminated or truncated:
                break
        for agent in agents:
            agent.decay_epsilon()
    return agents


In [57]:
pilot_policies = {}
for setting_name, setting in PILOT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        pilot_policies[(setting_name, training_seed)] = train_pilot_policy(
            setting, training_seed
        )
    print(f"Trained {setting_name} across five seeds")

Trained current across five seeds
Trained reduced_capacity across five seeds
Trained higher_cost across five seeds
Trained reduced_capacity_and_higher_cost across five seeds


## Evaluate on the same held-out starts

In [ ]:
def create_pilot_totals():
    return {
        "action_counts": [0, 0, 0, 0, 0],
        "pollution_values": [],
        "spillover_values": [],
        "regional_spreads": [],
        "remaining_emissions": 0.0,
        "mitigation_spending": 0.0,
    }


In [ ]:
def record_pilot_step(totals, info):
    totals["remaining_emissions"] += sum(info["remaining_emissions"])
    totals["mitigation_spending"] += sum(info["costs_paid"])
    totals["pollution_values"].extend(info["pollution_loads"])
    totals["spillover_values"].extend(info["incoming_spillover"])
    regional_spread = max(info["pollution_loads"]) - min(info["pollution_loads"])
    totals["regional_spreads"].append(regional_spread)


In [ ]:
def summarize_pilot_episode(
    environment, totals, setting_name, seed, training_seed, failed
):
    pollution_values = totals["pollution_values"]
    spillover_values = totals["spillover_values"]
    regional_spreads = totals["regional_spreads"]
    episode_region_steps = 4 * environment.step_count

    row = {
        "setting": setting_name,
        "training_seed": training_seed,
        "evaluation_seed": seed,
        "mean_pollution": sum(pollution_values) / len(pollution_values),
        "peak_pollution": max(pollution_values),
        "remaining_emissions": totals["remaining_emissions"],
        "mitigation_spending": totals["mitigation_spending"],
        "final_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "failed": failed,
        "mean_incoming_spillover": sum(spillover_values) / len(spillover_values),
        "mean_regional_spread": sum(regional_spreads) / len(regional_spreads),
    }

    action_names = ("save", "develop", "low", "medium", "high")
    for action_id, action_name in enumerate(action_names):
        row[f"{action_name}_share"] = (
            totals["action_counts"][action_id] / episode_region_steps
        )
    return row


In [ ]:
def evaluate_pilot_episode(agents, setting, seed, setting_name, training_seed):
    pilot_config = make_pilot_config(setting)
    environment = EconomicSustainabilityEnvironment(PILOT_REWARD_MODE, pilot_config)
    observations = environment.reset(seed)
    totals = create_pilot_totals()

    while True:
        actions = pilot_actions(agents, observations, greedy=True)
        for action in actions:
            totals["action_counts"][action] += 1

        observations, _, terminated, truncated, info = environment.step(actions)
        record_pilot_step(totals, info)

        if terminated or truncated:
            break

    return summarize_pilot_episode(
        environment, totals, setting_name, seed, training_seed, terminated
    )


In [ ]:
pilot_episode_rows = []
for setting_name, setting in PILOT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        agents = pilot_policies[(setting_name, training_seed)]
        for evaluation_seed in PILOT_EVALUATION_SEEDS:
            pilot_episode_rows.append(
                evaluate_pilot_episode(
                    agents, setting, evaluation_seed, setting_name, training_seed
                )
            )
pilot_episodes = pd.DataFrame(pilot_episode_rows)

PILOT_METRICS = [
    "mean_pollution", "peak_pollution", "remaining_emissions",
    "mitigation_spending", "final_edi", "final_currency",
    "failed", "mean_incoming_spillover", "save_share",
    "develop_share", "low_share", "medium_share", "high_share",
]
pilot_seed_results = pilot_episodes.groupby(
    ["setting", "training_seed"]
)[PILOT_METRICS].mean()
setting_results = pilot_seed_results.groupby(level="setting")



In [ ]:
display(setting_results[["mean_pollution", "peak_pollution", "remaining_emissions",
                       "mitigation_spending", "final_edi"]].agg(["mean", "std"]).round(2))


In [ ]:
display(setting_results[["final_currency", "failed", "mean_incoming_spillover"]]
        .agg(["mean", "std"]).round(2))


In [ ]:
display(setting_results[["save_share", "develop_share", "low_share",
                       "medium_share", "high_share"]].agg(["mean", "std"]).round(2))


## What the pilot suggests

- None of the four settings failed in the 400 held-out episodes. Even with the revised emissions rule, this pilot does not yet produce pollution near the failure threshold.
- With current capacity and cost, agents chose high mitigation 75% of the time. This setting had the lowest mean pollution (15.85) and remaining emissions (about 1,819), with mean final EDI 71.45. High mitigation capacity (30) can offset all new emissions at EDI up to 100 under the current formula, which helps explain this result.
- Raising prices by 1.5 times while keeping current capacities shifted action use toward medium mitigation: medium rose to 60% and high fell to 23%. Mean pollution increased to 22.98, remaining emissions to about 2,669, while final EDI stayed similar at 68.22. The higher price did not make the agents stop spending; total mitigation spending was slightly higher.
- Reducing capacities to 4, 8, and 12 lowered final EDI to about 41 and raised mean pollution to 24.91. Combining those lower capacities with higher costs lowered final EDI further, to about 36, and raised mean pollution to 36.98. Lower capacities leave more new emissions unavoided, even when agents choose mitigation often.
- These results make current high capacity look very generous in this model, while the reduced-capacity settings constrain both development and pollution control.

## Compare neighbor spillover levels

- The previous pilot changed mitigation capacity and cost while keeping neighbor spillover at 0.2. Here I hold current capacities and 1.5 times current mitigation costs fixed, and vary only spillover.
- I compare 0.0, the current 0.2, and 0.4. They are not real-word estimates.

In [ ]:
SPILLOVER_SETTINGS = {}
for spillover in (0.0, config.spillover, 0.4):
    setting_name = f"spillover_{spillover:.1f}"
    SPILLOVER_SETTINGS[setting_name] = {
        "capacities": config.mitigation_capacity,
        "costs": higher_action_costs,
        "spillover": spillover,
    }


In [61]:
spillover_policies = {}
for setting_name, setting in SPILLOVER_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        spillover_policies[(setting_name, training_seed)] = train_pilot_policy(
            setting, training_seed
        )
    print(f"Trained {setting_name} across five seeds")

Trained spillover_0.0 across five seeds
Trained spillover_0.2 across five seeds
Trained spillover_0.4 across five seeds


### Evaluate the spillover policies

In [ ]:
spillover_rows = []
for setting_name, setting in SPILLOVER_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        agents = spillover_policies[(setting_name, training_seed)]
        for evaluation_seed in PILOT_EVALUATION_SEEDS:
            spillover_rows.append(
                evaluate_pilot_episode(
                    agents, setting, evaluation_seed, setting_name, training_seed
                )
            )
spillover_episodes = pd.DataFrame(spillover_rows)
spillover_seed_results = spillover_episodes.groupby(
    ["setting", "training_seed"]
)[PILOT_METRICS + ["mean_regional_spread"]].mean()
spillover_summary = spillover_seed_results.groupby(level="setting")


In [ ]:
display(spillover_summary[["mean_pollution", "peak_pollution",
                           "remaining_emissions", "mitigation_spending",
                           "final_edi", "failed"]]
        .agg(["mean", "std"]).round(2))


In [ ]:
display(spillover_summary[["mean_incoming_spillover",
                           "mean_regional_spread"]]
        .agg(["mean", "std"]).round(2))


In [ ]:
display(spillover_summary[["low_share", "medium_share", "high_share"]]
        .agg(["mean", "std"]).round(2))


## What this comparison suggests

- Mean incoming spillover rises from 0 at spillover 0.0 to 1.33 at 0.2 and 1.93 at 0.4. Regional pollution levels become somewhat more alike as spillover increases: mean regional spread falls from 20.83 at 0.0 to 17.06 at 0.4.
- Mean pollution and peak pollution do not rise steadily with spillover. Mean pollution is 15.38, 22.98, and 21.50 at 0.0, 0.2, and 0.4; peak pollution is 49.13, 57.48, and 55.90. At 0.4, agents also ended with lower EDI and emissions than at 0.2, which can offset the extra pollution they receive from neighbors.
- No episode failed at any spillover setting. Because each setting was trained separately, this comparison mixes the direct effect of pollution sharing with policy adaptation. To isolate the direct effect, a follow-up could evaluate every trained policy under all three spillover settings.
- No evaluation episode failed at any tested spillover. These results do not show that 0.2 is optimal or that stronger spillover is realistic.